In [ ]:
""" Load_raw_to_deltatable, Reads Parquet from Lakehouse landing zone and writes Delta tables."""

import base64,glob, logging, os,yaml
from datetime import datetime
from pathlib import Path
from typing import Any
import pandas as pd
import pyarrow as pa


try:
    from deltalake import DeltaTable, write_deltalake
except ImportError as exc:
    raise ImportError(
        "Package 'deltalake' is required for Python-kernel Delta writes. "
        "Install it in the Fabric environment, for example: %pip install deltalake"
    ) from exc

try:
    from python_utils.text_cleaning import clean_text_columns_df
except Exception:
    try:
        from cdp_spark_utils.text_cleaning import clean_text_columns_df
    except Exception:
        # Local notebook fallback when text_cleaning.py is uploaded next to the notebook.
        from text_cleaning import clean_text_columns_df

logging.basicConfig(
    level=os.getenv("LOG_LEVEL", "INFO"),
    format="%(asctime)s %(levelname)s [%(name)s] %(message)s",
    force=True,
)
logger = logging.getLogger(__name__)

LAKEHOUSE_ROOT = Path(os.getenv("LAKEHOUSE_ROOT", "/lakehouse/default"))
LANDING_ROOT = LAKEHOUSE_ROOT / "Files" / "landing"
TABLES_ROOT = LAKEHOUSE_ROOT / "Tables" / "raw"
TABLES_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
try:
    param_string_b64
except NameError:
    param_string_b64 = ""

# Local test example:
# test_config_yaml = """
# source_system: JSONplaceholder
# execution_date: "2026-05-15"
# tables:
#   - source_table: posts
#     load_type: merge
#     merge_columns: postId
#     activated: true
#     text_clean_columns: [Symptom]
# """
# param_string_b64 = base64.urlsafe_b64encode(test_config_yaml.encode("utf-8")).decode("utf-8")


In [ ]:
def decode_config_yaml(param_string_b64: str) -> dict[str, Any]:
    payload = (param_string_b64 or "").strip()
    if not payload:
        raise ValueError("Parameter 'param_string_b64' is empty.")

    for decoder in (base64.urlsafe_b64decode, base64.b64decode):
        try:
            decoded = decoder(payload.encode("utf-8")).decode("utf-8")
            config = yaml.safe_load(decoded)
            if isinstance(config, dict):
                return config
        except Exception:
            pass

    config = yaml.safe_load(payload)
    if not isinstance(config, dict):
        raise ValueError("param_string_b64 must contain valid YAML config, base64 encoded or plain YAML.")
    return config


def is_activated(value: Any) -> bool:
    return str(value).strip().lower() == "true" if not isinstance(value, bool) else value


def as_list(value: Any) -> list[str]:
    if value is None or value == "":
        return []
    if isinstance(value, list):
        return [str(v) for v in value]
    return [str(value)]


def read_parquet_files(pattern: str) -> pd.DataFrame | None:
    files = sorted(glob.glob(pattern, recursive=True))
    if not files:
        return None
    frames = [pd.read_parquet(file) for file in files]
    return pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]


def add_metadata_columns(df: pd.DataFrame, metadata: dict[str, Any]) -> pd.DataFrame:
    result = df.copy()
    for col, value in metadata.items():
        if value is not None:
            result[col] = str(value)
    return result


def normalize_for_delta(df: pd.DataFrame) -> pd.DataFrame:
    # Keep complex/untyped pandas objects as strings so pyarrow/delta-rs can write consistently.
    result = df.copy()
    for col in result.columns:
        if result[col].dtype == "object":
            result[col] = result[col].map(lambda x: None if pd.isna(x) else str(x))
    return result


def table_path(table_name: str) -> Path:
    return TABLES_ROOT / table_name


def delta_exists(path: Path) -> bool:
    return (path / "_delta_log").exists()


def write_full_or_append(df: pd.DataFrame, path: Path, mode: str) -> None:
    write_deltalake(
        str(path),
        pa.Table.from_pandas(df, preserve_index=False),
        mode=mode,
        schema_mode="merge",
    )


def merge_delta(df: pd.DataFrame, path: Path, merge_columns: list[str]) -> None:
    if not merge_columns:
        raise ValueError("merge_columns is required for load_type='merge'.")

    missing = [col for col in merge_columns if col not in df.columns]
    if missing:
        raise ValueError(f"Merge columns {missing} not found. Available columns: {list(df.columns)}")

    dt = DeltaTable(str(path))
    source_table = pa.Table.from_pandas(df, preserve_index=False)
    predicate = " AND ".join([f"target.{col} = source.{col}" for col in merge_columns])

    (
        dt.merge(
            source=source_table,
            predicate=predicate,
            source_alias="source",
            target_alias="target",
        )
        .when_matched_update_all()
        .when_not_matched_insert_all()
        .execute()
    )


def upsert_delta_table(df_source: pd.DataFrame, load_type: str, target_table_name: str, merge_columns: list[str]) -> None:
    path = table_path(target_table_name)
    path.mkdir(parents=True, exist_ok=True)
    df_source = normalize_for_delta(df_source)

    exists = delta_exists(path)
    load_type = (load_type or "append").lower()
    logger.info("Table path %s exists: %s", path, exists)

    if not exists or load_type in ["full", "backfill"]:
        logger.info("Creating/overwriting Delta table %s", target_table_name)
        write_full_or_append(df_source, path, mode="overwrite")
    elif load_type == "append":
        logger.info("Appending to Delta table %s", target_table_name)
        write_full_or_append(df_source, path, mode="append")
    elif load_type == "merge":
        logger.info("Merging into Delta table %s using columns %s", target_table_name, merge_columns)
        merge_delta(df_source, path, merge_columns)
    else:
        raise ValueError("Unknown load_type. Supported values: full, backfill, append, merge.")

    logger.info("Completed load for %s with %s rows", target_table_name, len(df_source))


In [ ]:
configuration = decode_config_yaml(param_string_b64)
tables = configuration.get("tables", [])

source_system = str(configuration.get("source_system", "")).lower().strip()
if not source_system:
    raise ValueError("source_system is required.")

sub_source_system = configuration.get("sub_source_system")
sub_source_system = str(sub_source_system).strip() if sub_source_system else None
base_url = configuration.get("base_url")

lakehouse_base_path = f"{source_system}/{sub_source_system}" if sub_source_system else source_system
source_database = sub_source_system or source_system

execution_date_str = str(configuration.get("execution_date", "")).strip()[:10]
execution_date = datetime.strptime(execution_date_str, "%Y-%m-%d")
year, month, day = execution_date.year, execution_date.month, execution_date.day

metadata_base = {
    "source_system": source_system,
    "execution_date": execution_date_str,
    "execution_timestamp": configuration.get("execution_timestamp"),
    "data_interval_start": configuration.get("data_interval_start"),
    "data_interval_end": configuration.get("data_interval_end"),
    "dag_run_id": configuration.get("dag_run_id"),
    "dag_id": configuration.get("dag_id"),
    "base_url": base_url,
    "source_database": source_database,
}

for table in tables:
    if not is_activated(table.get("activated")):
        continue

    source_table = str(table.get("source_table", "")).strip()
    load_type = str(table.get("load_type", "append")).lower().strip()
    merge_columns = as_list(table.get("merge_columns"))
    text_clean_columns = as_list(table.get("text_clean_columns"))
    target_table_name = f"{source_system}_{source_table}"

    logger.info("Processing table: %s", source_table)
    logger.info("Target Delta path: %s", table_path(target_table_name))
    logger.info("Load type: %s", load_type)

    if load_type == "backfill":
        source_pattern = str(LANDING_ROOT / lakehouse_base_path / source_table / "**" / "*.parquet")
    elif load_type in ["full", "append", "merge"]:
        source_pattern = str(LANDING_ROOT / lakehouse_base_path / source_table / str(year) / f"{month:02d}" / f"{day:02d}" / "*.parquet")
    else:
        logger.error("Incorrect load_type provided: %s. Skipping table.", load_type)
        continue

    logger.info("Reading parquet from: %s", source_pattern)
    df_source = read_parquet_files(source_pattern)
    if df_source is None or df_source.empty:
        logger.warning("No parquet rows found for path: %s. Skipping table.", source_pattern)
        continue

    logger.info("Source records: %s", len(df_source))
    # Delete duplicated rows
    dedup_columns = [col for col in df_source.columns if col not in ["source_database", "sub_source_system"]]
    df_deduplicated = df_source.drop_duplicates(subset=dedup_columns, ignore_index=True)
    logger.info("After deduplication: %s records", len(df_deduplicated))
    # Clean Web tags html, css .. from text_clean_columns config-property
    df_cleaned = df_deduplicated
    if text_clean_columns:
        df_cleaned = clean_text_columns_df(df_cleaned, text_clean_columns)
    # Adding sub_source_system for each unit
    metadata = dict(metadata_base)
    if sub_source_system:
        metadata["sub_source_system"] = sub_source_system
        if merge_columns:
            merge_columns = merge_columns + ["sub_source_system"]

    df_metadata = add_metadata_columns(df_cleaned, metadata)
    logger.info("Final columns: %s", list(df_metadata.columns))

    upsert_delta_table(df_metadata, load_type, target_table_name, merge_columns)
